# ATAC analysis for CD4 T cells in preRA-TEASEQ
- call peaks and calcualte sampletile matrix for all cd4 t cells

# Setup

In [1]:
quiet_library <- function(...) {
    suppressPackageStartupMessages(library(...))
}
quiet_library('Seurat')
quiet_library('tidyverse')
quiet_library('future')
quiet_library('future.apply')
quiet_library("ArchR")
quiet_library("rstatix")
quiet_library("chromVAR")



                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [2]:
# Load scMACS and accompanying libraries
quiet_library(MOCHA)
quiet_library(data.table)
quiet_library(GenomicRanges)
quiet_library(plyranges)


In [3]:
packageVersion("MOCHA")


[1] ‘1.1.0’

In [4]:
# Define your annotation package for TxDb object(s)
# and genome-wide annotation
# Here our samples are human using hg38 as a reference.
# For more info: https://bioconductor.org/packages/3.15/data/annotation/
library(TxDb.Hsapiens.UCSC.hg38.refGene)
library(BSgenome.Hsapiens.UCSC.hg38)
library(org.Hs.eg.db)
TxDb <- TxDb.Hsapiens.UCSC.hg38.refGene
Org <- org.Hs.eg.db


Loading required package: GenomicFeatures

Loading required package: AnnotationDbi


Attaching package: ‘AnnotationDbi’


The following object is masked from ‘package:plyranges’:

    select


The following object is masked from ‘package:rstatix’:

    select


The following object is masked from ‘package:dplyr’:

    select


Loading required package: BSgenome

Loading required package: Biostrings

Loading required package: XVector


Attaching package: ‘XVector’


The following object is masked from ‘package:plyr’:

    compact


The following object is masked from ‘package:purrr’:

    compact



Attaching package: ‘Biostrings’


The following object is masked from ‘package:grid’:

    pattern


The following object is masked from ‘package:base’:

    strsplit


Loading required package: BiocIO

Loading required package: rtracklayer


Attaching package: ‘rtracklayer’


The following object is masked from ‘package:BiocIO’:

    FileForFormat






In [5]:
# Check number of cores
future::availableCores()
# Set up parallel processing to run when using 'future' functions
future::plan(strategy = "multicore", workers = future::availableCores() - 3)
options(future.globals.maxSize = 1000 * 1024^5)
# to turn off parallel processing run line below
# future::plan(strategy = "sequential")


system 
    16

In [6]:
# define working path
data_path <- "/home/jupyter/data/preRA_teaseq/EXP-00243"
meta_path <- "/home/jupyter/data/preRA_teaseq/meta_data"
output_path <- "/home/jupyter/data/preRA_teaseq/output_results/cd4_t/atac"
fig_path <- "/home/jupyter/data/preRA_teaseq/EXP-00243/cd4t_cells/Plots"
if (!dir.exists(fig_path)) (dir.create(fig_path, recursive = TRUE))
if (!dir.exists(output_path)) (dir.create(output_path, recursive = TRUE))
# define a project name
proj_name <- "PreRA_teaseq_cd4_t_atac"


In [7]:
# define the color palette to be used
npg_color <- c(
    "#E64B35FF", "#4DBBD5FF", "#00A087FF", "#3C5488FF", "#F39B7FFF",
    "#8491B4FF", "#91D1C2FF", "#DC0000FF", "#7E6148FF", "#B09C85FF"
)
nejm_color <- c("#BC3C29FF", "#0072B5FF", "#E18727FF", "#20854EFF", "#7876B1FF", "#6F99ADFF", "#FFDC91FF", "#EE4C97FF")
jama_color <- c("#374E55FF", "#DF8F44FF", "#00A1D5FF", "#B24745FF", "#79AF97FF", "#6A6599FF", "#80796BFF")
jco_color <- c("#0073C2FF", "#EFC000FF", "#868686FF", "#CD534CFF", "#7AA6DCFF", "#003C67FF", "#8F7700FF")
cluster_colors <- c(
    "#DC050C", "#FB8072", "#1965B0", "#7BAFDE", "#882E72", "#B17BA6", "#FF7F00", "#FDB462", "#E7298A",
    "#E78AC3", "#33A02C", "#B2DF8A", "#55A1B1", "#8DD3C7", "#A6761D", "#E6AB02", "#7570B3", "#BEAED4", "#666666", "#999999",
    "#aa8282", "#d4b7b7", "#8600bf", "#ba5ce3", "#808000", "#aeae5c", "#1e90ff", "#00bfff", "#56ff0d", "#ffff00"
)
cluster_colors_ext <- colorRampPalette(cluster_colors)(36)
options(repr.plot.width = 20, repr.plot.height = 15)


In [8]:
source("/home/jupyter/github/Teaseq-analysis/scRNA_teaseq_ananlysis_helper_functions.r")


In [9]:
# pathway analysis with enrichR
quiet_library(enrichR)
setEnrichrSite("Enrichr")
dbs <- listEnrichrDbs()
# %% codecell|


Connection changed to https://maayanlab.cloud/Enrichr/

Connection is Live!



## Load data

In [10]:
# load the seurat object
# ra_tea_cd4 <- readRDS(file.path(data_path, 'PreRA_teaseq_seurat_cd4_t_filtered_cells_rmBR2024.rds'))


In [11]:
# set ArchR parameters
addArchRThreads(threads = 55)
addArchRGenome("hg38")
set.seed(1221)


Input threads is equal to or greater than ncores minus 1 (15)
Setting cores to ncores minus 2. Set force = TRUE to set above this number!

Setting default number of Parallel threads to 14.

Setting default genome to Hg38.



In [22]:
# load ATAC
# load the ArchR project for CD4 t cells
cd4t_atac <- loadArchRProject(path = "/home/jupyter/data/preRA_teaseq/EXP-00243/cd4t_cells")


Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [23]:
cd4t_atac



           ___      .______        ______  __    __  .______      
          /   \     |   _  \      /      ||  |  |  | |   _  \     
         /  ^  \    |  |_)  |    |  ,----'|  |__|  | |  |_)  |    
        /  /_\  \   |      /     |  |     |   __   | |      /     
       /  _____  \  |  |\  \\___ |  `----.|  |  |  | |  |\  \\___.
      /__/     \__\ | _| `._____| \______||__|  |__| | _| `._____|
    



class: ArchRProject 
outputDirectory: /home/jupyter/data/preRA_teaseq/EXP-00243/cd4t_cells 
samples(13): EXP-00243-P1_PB00067-03 EXP-00243-P1_PB00073-03 ...
  EXP-00243-P1_PB00501-03 EXP-00243-P1_PB00212-03
sampleColData names(1): ArrowFiles
cellColData names(51): barcodes Sample ... Clusters_0.8 CD4TnaC2toC1
numberOfCells(1): 44806
medianTSS(1): 26.517
medianFrags(1): 5364

In [24]:
# source('/home/jupyter/github/scATAC_Supplements/ArchR_Supplements.R')
# source('/home/jupyter/github/scATAC_Supplements/AlternativeTSS_USage.R')


## peak calls in the all cd4

In [25]:
table(cd4t_atac$clean_l2_cell_types)



        cd4_ctl      cd4_memory       cd4_naive  cd4_temra_like      cd8_memory 
            245           17052           22954               9            1432 
      cd8_naive       cd8_temra         cd8_trm            dn_t            gd_t 
            110             108               9              73              35 
           mait    nk_like_t_nk t_proliferating            treg 
            187              13              35            2544 

In [ ]:
# # call peaks in total CD4 naive cells
# cellPopulations <- cd4t_atac$clean_l2_cell_types %>% unique()
# cellPopLabel <- "t_anno_atac"
numCores <- 59
# Parameters for generating the sample-tile matrices
threshold <- 0.2
groupColumn <- "cohort"
join <- "union"


In [ ]:
cellPopLabel <- "clean_l2_cell_types"
cd4t_atac$cohort %>% unique()
cd4t_atac$clean_l2_cell_types %>% unique()
cellPopulations <- c("cd4_naive", "cd4_memory", "treg")
# cellPopulations <- cd4t_atac$Clusters_0.8 %>% unique()
# cellPopulations = cellPopulations[cellPopulations!='C1']
# cellPopulations


In [ ]:
# cellPopulations
# ?callOpenTiles


In [ ]:
####################################################
# 2. Call open tiles (main peak calling step)
#    Done once for all specified cell populations
####################################################
cd4_tileResults <- MOCHA::callOpenTiles(
    cd4t_atac,
    cellPopLabel = cellPopLabel,
    cellPopulations = cellPopulations,
    TxDb = "TxDb.Hsapiens.UCSC.hg38.refGene",
    Org = "org.Hs.eg.db",
    numCores = numCores,
    outDir = output_path
)


In [ ]:
cd4_tileResults


In [ ]:
# save the tile matrix from mocha
cd4_tileResults %>% saveRDS(file.path(
    output_path,
    paste0(proj_name, "_MOCHA_clean_l2_cell_types_tiles_matrix.rds")
))


In [ ]:
# load the tile matrix from mocha
cd4_tileResults <- readRDS(file.path(
    output_path,
    paste0(proj_name, "_MOCHA_clean_l2_cell_types_tiles_matrix.rds")
))


In [ ]:
plotConsensus(
  tileObject,
  cellPopulations = "All",
  groupColumn = NULL,
  returnPlotList = FALSE,
  returnDFs = FALSE,
  numCores = 1
)


In [ ]:
# call sample tile matrix
SampleTileMatrices <- MOCHA::getSampleTileMatrix(cd4_tileResults,
    numCores = numCores,
    cellPopulations = cellPopulations, groupColumn = groupColumn,
    threshold = threshold, verbose = TRUE
)


In [ ]:
SampleTileMatrices


In [ ]:
# This function can also take any GRanges object
# and add annotations to its metadata.
SampleTileMatricesAnnotated <- MOCHA::annotateTiles(SampleTileMatrices)

# Load a curated motif set from library(chromVARmotifs)
# included with ArchR installation
library(chromVARmotifs)
data(human_pwms_v2)

SampleTileMatricesAnnotated <- MOCHA::addMotifSet(
  SampleTileMatricesAnnotated,
  pwms = human_pwms_v2,
  w = 7 # weight parameter for motifmatchr
)


In [ ]:
# save the sample tile martrix
SampleTileMatricesAnnotated %>% saveRDS(file.path(
    output_path,
    paste0(proj_name, "_MOCHA_total_clean_l2_cell_types_SampleTileMatrices.rds")
))


In [5]:
# # upload data to hise
# hise::uploadFiles(
#   files = list('/home/workspace/data/preRA_teaseq/output_results/cd4_t/atac/PreRA_teaseq_cd4_t_atac_MOCHA_total_clean_l2_cell_types_SampleTileMatrices.rds'),
#   studySpaceId = '72ac97bb-a08c-4d89-8180-e4d057be2c70',
#   title = 'preRA TeaSeq CD4 T MOCHA SampleTileMatrix',
#   inputFileIds = list("04464893-3078-4e22-8b08-a16890cda338"),
#   doPrompt = TRUE
# )

In [6]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/minimal/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.6.5       httr_1.4.7        cli_3.6.3        
 [5] rlang_1.1.4       stringi_1.8.7     generics_0.1.4    assertthat_0.2.1 
 [9] jsonlite_1.8.9    glue_1.8.0        RCurl_1.98-1.16   htmltools_0.5.8.1
[13] IRdisplay_1.1     IRkernel_1.3.2    fansi_1.0.6       tibble_3.2.1 